# CSV Label Profile Generator

This notebook reads a CSV file, excludes company/id/address/url/file fields, extracts categorical unique values, and summarises numeric/date ranges into an Excel workbook.

Default output directory: `E:\\000硕士毕设\\公司选取`.

## 1. Configuration

Set `INPUT_CSV` before running. If `OUTPUT_XLSX_NAME` is left as `None`, the Excel file name will be generated from the CSV file name.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import re
import numpy as np
import pandas as pd

# Option A: set this to your CSV path.
# Option B: leave it as None, and the notebook will auto-use the only CSV found in OUTPUT_DIR.
INPUT_CSV = None

OUTPUT_DIR = Path(r"E:\\000硕士毕设\\公司选取")
OUTPUT_XLSX_NAME = None  # e.g. "company_selection_label_profile.xlsx"

CSV_ENCODING = "utf-8-sig"
CHUNKSIZE = 200_000
SAMPLE_ROWS_FOR_TYPE_INFERENCE = 20_000

# Confirmed rules
HIGH_CARDINALITY_THRESHOLD = 1000
HIGH_CARDINALITY_SAMPLE_TOP_N = 50
NUMERIC_PARSE_RATIO = 0.95
DATE_PARSE_RATIO = 0.80

# Store numeric samples for approximate percentiles. Min/max always use all rows.
MAX_NUMERIC_SAMPLE_VALUES_PER_COLUMN = 1_000_000

EXCLUDE_KEYWORDS = [
    "companynumber", "company_number", "companyno", "company_no",
    "companyname", "company_name", "company", "businessname", "business_name",
    "id", "identifier", "uuid", "guid", "uri", "url", "link",
    "filename", "file_name", "internal_filename", "source_file", "source_zip",
    "address", "postcode", "post_code", "zipcode", "zip_code",
    "regaddress", "registeredoffice", "careof", "po_box",
]

BOOLEAN_VALUES = {"true", "false", "yes", "no", "y", "n", "0", "1"}
DATE_NAME_HINTS = ["date", "month", "year", "period", "incorporation", "madeup", "made_up", "created", "updated"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_CSV is None or str(INPUT_CSV).strip() == "" or "your_input_file.csv" in str(INPUT_CSV):
    csv_candidates = sorted(OUTPUT_DIR.glob("*.csv"))
    if len(csv_candidates) == 1:
        input_path = csv_candidates[0]
        print("Auto-selected the only CSV in OUTPUT_DIR:", input_path)
    else:
        print("CSV files found in OUTPUT_DIR:")
        for p in csv_candidates[:50]:
            print(" -", p)
        raise ValueError(
            "Please set INPUT_CSV to the exact CSV path. "
            f"Found {len(csv_candidates)} CSV files in {OUTPUT_DIR}."
        )
else:
    input_path = Path(INPUT_CSV)

if not input_path.exists():
    raise FileNotFoundError(
        f"CSV file not found: {input_path}. Please update INPUT_CSV in the configuration cell."
    )
if OUTPUT_XLSX_NAME is None:
    OUTPUT_XLSX = OUTPUT_DIR / f"{input_path.stem}_label_profile.xlsx"
else:
    OUTPUT_XLSX = OUTPUT_DIR / OUTPUT_XLSX_NAME

print("Input CSV:", input_path)
print("Output Excel:", OUTPUT_XLSX)

Auto-selected the only CSV in OUTPUT_DIR: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned.csv
Input CSV: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned.csv
Output Excel: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned_label_profile.xlsx


## 2. Helper Functions

In [2]:
def normalise_colname(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def exclusion_reason(column: str):
    key = normalise_colname(column)
    for pattern in EXCLUDE_KEYWORDS:
        pattern_key = normalise_colname(pattern)
        if pattern_key and pattern_key in key:
            return f"excluded_by_name_keyword:{pattern}"
    return None


def clean_string_series(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()


def to_numeric_series(s: pd.Series) -> pd.Series:
    cleaned = clean_string_series(s).str.replace(",", "", regex=False)
    cleaned = cleaned.str.replace("£", "", regex=False).str.replace("$", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")


def parse_date_series(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)


def infer_column_kind(column: str, sample_s: pd.Series) -> str:
    if exclusion_reason(column):
        return "excluded"

    non_null = sample_s.dropna()
    if non_null.empty:
        return "empty"

    text = clean_string_series(non_null)
    text = text[text.notna() & (text != "")]
    if text.empty:
        return "empty"

    lower_values = set(text.str.lower().dropna().unique().tolist())
    if len(lower_values) <= 2 and lower_values.issubset(BOOLEAN_VALUES):
        return "binary"

    numeric = to_numeric_series(text)
    numeric_ratio = numeric.notna().mean() if len(text) else 0
    if numeric_ratio >= NUMERIC_PARSE_RATIO:
        return "numeric"

    col_key = normalise_colname(column)
    date_name_hint = any(hint in col_key for hint in DATE_NAME_HINTS)
    parsed_dates = parse_date_series(text)
    date_ratio = parsed_dates.notna().mean() if len(text) else 0
    if date_name_hint and date_ratio >= 0.50:
        return "date"
    if date_ratio >= DATE_PARSE_RATIO and text.nunique(dropna=True) > 5:
        return "date"

    return "categorical"


def safe_sheet_name(name: str) -> str:
    return re.sub(r"[\\/*?:\[\]]", "_", name)[:31]


def add_numeric_sample(existing_parts, new_values: pd.Series, current_count: int):
    values = new_values.dropna().to_numpy(dtype=float)
    if len(values) == 0 or current_count >= MAX_NUMERIC_SAMPLE_VALUES_PER_COLUMN:
        return existing_parts, current_count
    remaining = MAX_NUMERIC_SAMPLE_VALUES_PER_COLUMN - current_count
    if len(values) > remaining:
        rng = np.random.default_rng(42 + current_count)
        values = rng.choice(values, size=remaining, replace=False)
    existing_parts.append(values)
    return existing_parts, current_count + len(values)

## 3. Infer Column Types

In [3]:
header = pd.read_csv(input_path, nrows=0, encoding=CSV_ENCODING)
columns = list(header.columns)

sample = pd.read_csv(
    input_path,
    nrows=SAMPLE_ROWS_FOR_TYPE_INFERENCE,
    encoding=CSV_ENCODING,
    low_memory=False,
)

column_kinds = {}
excluded_reasons = {}
for col in columns:
    reason = exclusion_reason(col)
    if reason:
        column_kinds[col] = "excluded"
        excluded_reasons[col] = reason
    else:
        column_kinds[col] = infer_column_kind(col, sample[col] if col in sample.columns else pd.Series(dtype="object"))

column_overview_initial = pd.DataFrame({
    "column_name": columns,
    "inferred_kind": [column_kinds[c] for c in columns],
    "excluded_reason": [excluded_reasons.get(c, "") for c in columns],
})
column_overview_initial["inferred_kind"].value_counts(dropna=False)

C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a fu

inferred_kind
excluded       11
categorical     6
date            3
numeric         3
binary          2
Name: count, dtype: int64

## 4. Scan Full CSV

In [4]:
total_rows = 0
non_null_counts = defaultdict(int)
missing_counts = defaultdict(int)
unique_counters = defaultdict(Counter)
high_cardinality_flags = defaultdict(bool)

numeric_stats = defaultdict(lambda: {
    "count": 0, "missing": 0, "min": np.inf, "max": -np.inf,
    "sum": 0.0, "sum_sq": 0.0, "zero_count": 0, "negative_count": 0,
})
numeric_samples = defaultdict(list)
numeric_sample_counts = defaultdict(int)

date_stats = defaultdict(lambda: {"count": 0, "missing": 0, "min": pd.NaT, "max": pd.NaT, "years": Counter()})

tracked_columns = [c for c in columns if column_kinds[c] != "excluded"]

for i, chunk in enumerate(pd.read_csv(input_path, chunksize=CHUNKSIZE, encoding=CSV_ENCODING, low_memory=False), start=1):
    total_rows += len(chunk)
    print(f"Processing chunk {i:,}; rows scanned: {total_rows:,}")

    for col in tracked_columns:
        kind = column_kinds[col]
        s = chunk[col]
        missing = s.isna().sum() + (clean_string_series(s).eq("").sum() if kind in {"categorical", "binary", "date", "empty"} else 0)
        missing_counts[col] += int(missing)
        non_null_counts[col] += int(len(s) - missing)

        if kind == "numeric":
            values = to_numeric_series(s)
            valid = values.dropna()
            st = numeric_stats[col]
            st["count"] += int(valid.shape[0])
            st["missing"] += int(values.isna().sum())
            if not valid.empty:
                st["min"] = min(st["min"], float(valid.min()))
                st["max"] = max(st["max"], float(valid.max()))
                st["sum"] += float(valid.sum())
                st["sum_sq"] += float((valid ** 2).sum())
                st["zero_count"] += int((valid == 0).sum())
                st["negative_count"] += int((valid < 0).sum())
                numeric_samples[col], numeric_sample_counts[col] = add_numeric_sample(
                    numeric_samples[col], valid, numeric_sample_counts[col]
                )

        elif kind == "date":
            dates = parse_date_series(s)
            valid = dates.dropna()
            st = date_stats[col]
            st["count"] += int(valid.shape[0])
            st["missing"] += int(dates.isna().sum())
            if not valid.empty:
                current_min = valid.min()
                current_max = valid.max()
                st["min"] = current_min if pd.isna(st["min"]) else min(st["min"], current_min)
                st["max"] = current_max if pd.isna(st["max"]) else max(st["max"], current_max)
                st["years"].update(valid.dt.year.dropna().astype(int).astype(str).tolist())

        elif kind in {"categorical", "binary", "empty"}:
            values = clean_string_series(s).dropna()
            values = values[values != ""]
            if not values.empty:
                unique_counters[col].update(values.tolist())
                if len(unique_counters[col]) > HIGH_CARDINALITY_THRESHOLD:
                    high_cardinality_flags[col] = True

print(f"Done. Total rows scanned: {total_rows:,}")

Processing chunk 1; rows scanned: 200,000


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a fu

Processing chunk 2; rows scanned: 400,000


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a fu

Processing chunk 3; rows scanned: 600,000


...

Processing chunk 17; rows scanned: 3,400,000


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a fu

Processing chunk 18; rows scanned: 3,415,689


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_9076\1541134854.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a fu

Done. Total rows scanned: 3,415,689


## 5. Build Output Tables

In [5]:
overview_rows = []
for col in columns:
    kind = column_kinds[col]
    overview_rows.append({
        "column_name": col,
        "kind": kind,
        "non_null_count": non_null_counts.get(col, "") if kind != "excluded" else "",
        "missing_count": missing_counts.get(col, "") if kind != "excluded" else "",
        "unique_count": len(unique_counters.get(col, {})) if kind in {"categorical", "binary", "empty"} else "",
        "is_high_cardinality": bool(high_cardinality_flags.get(col, False)),
        "excluded_reason": excluded_reasons.get(col, ""),
    })
column_overview = pd.DataFrame(overview_rows)

categorical_rows = []
categorical_summary_rows = []
high_cardinality_rows = []
binary_rows = []

for col, counter in unique_counters.items():
    kind = column_kinds[col]
    unique_count = len(counter)
    total_non_null = sum(counter.values())
    common_values = counter.most_common()
    categorical_summary_rows.append({
        "column_name": col,
        "kind": kind,
        "unique_count": unique_count,
        "non_null_count": total_non_null,
        "missing_count": missing_counts.get(col, 0),
        "is_high_cardinality": unique_count > HIGH_CARDINALITY_THRESHOLD,
        "top_values_preview": "; ".join([f"{v} ({c})" for v, c in common_values[:10]]),
    })

    if kind == "binary":
        for value, count in common_values:
            binary_rows.append({
                "column_name": col,
                "value": value,
                "count": count,
                "percent_of_non_null": count / total_non_null if total_non_null else np.nan,
            })
    elif unique_count <= HIGH_CARDINALITY_THRESHOLD:
        for value, count in common_values:
            categorical_rows.append({
                "column_name": col,
                "value": value,
                "count": count,
                "percent_of_non_null": count / total_non_null if total_non_null else np.nan,
                "dtype_group": kind,
            })
    else:
        for rank, (value, count) in enumerate(common_values[:HIGH_CARDINALITY_SAMPLE_TOP_N], start=1):
            high_cardinality_rows.append({
                "column_name": col,
                "unique_count": unique_count,
                "sample_rank_by_frequency": rank,
                "sample_value": value,
                "sample_count": count,
                "note": f"unique_count>{HIGH_CARDINALITY_THRESHOLD}; top {HIGH_CARDINALITY_SAMPLE_TOP_N} values only",
            })

numeric_rows = []
for col, st in numeric_stats.items():
    count = st["count"]
    mean = st["sum"] / count if count else np.nan
    variance = (st["sum_sq"] / count - mean ** 2) if count else np.nan
    std = math.sqrt(max(variance, 0)) if count else np.nan
    sample_values = np.concatenate(numeric_samples[col]) if numeric_samples[col] else np.array([])
    percentiles = {}
    if len(sample_values):
        for p in [1, 5, 25, 50, 75, 95, 99]:
            percentiles[f"p{p}"] = float(np.percentile(sample_values, p))
    else:
        for p in [1, 5, 25, 50, 75, 95, 99]:
            percentiles[f"p{p}"] = np.nan
    numeric_rows.append({
        "column_name": col,
        "count": count,
        "missing_count": st["missing"],
        "missing_percent": st["missing"] / total_rows if total_rows else np.nan,
        "min": st["min"] if st["min"] != np.inf else np.nan,
        **percentiles,
        "max": st["max"] if st["max"] != -np.inf else np.nan,
        "mean": mean,
        "std": std,
        "zero_count": st["zero_count"],
        "negative_count": st["negative_count"],
        "percentiles_note": "percentiles are exact if value count <= sample cap; otherwise sampled; min/max are full-scan exact",
    })

date_rows = []
for col, st in date_stats.items():
    years = ", ".join([year for year, _ in st["years"].most_common()])
    date_rows.append({
        "column_name": col,
        "count": st["count"],
        "missing_count": st["missing"],
        "missing_percent": st["missing"] / total_rows if total_rows else np.nan,
        "min_date": st["min"],
        "max_date": st["max"],
        "unique_years": years,
    })

excluded_columns = pd.DataFrame([
    {"column_name": col, "excluded_reason": reason}
    for col, reason in excluded_reasons.items()
])

readme = pd.DataFrame([
    {"item": "input_csv", "value": str(input_path)},
    {"item": "output_xlsx", "value": str(OUTPUT_XLSX)},
    {"item": "total_rows_scanned", "value": total_rows},
    {"item": "high_cardinality_threshold", "value": HIGH_CARDINALITY_THRESHOLD},
    {"item": "excluded_fields", "value": "company/id/address/postcode/url/filename/source fields are excluded by name keyword"},
    {"item": "categorical_unique_values", "value": "one row per unique value if unique_count <= threshold"},
    {"item": "numeric_summary", "value": "full-scan min/max; percentile values may be sampled for very large columns"},
])

categorical_unique_values = pd.DataFrame(categorical_rows)
categorical_summary = pd.DataFrame(categorical_summary_rows)
numeric_summary = pd.DataFrame(numeric_rows)
date_summary = pd.DataFrame(date_rows)
binary_columns = pd.DataFrame(binary_rows)
high_cardinality_columns = pd.DataFrame(high_cardinality_rows)

print("Tables prepared:")
for name, df in [
    ("column_overview", column_overview),
    ("categorical_unique_values", categorical_unique_values),
    ("categorical_summary", categorical_summary),
    ("numeric_summary", numeric_summary),
    ("date_summary", date_summary),
    ("binary_columns", binary_columns),
    ("high_cardinality_columns", high_cardinality_columns),
    ("excluded_columns", excluded_columns),
]:
    print(f"{name}: {len(df):,} rows")

Tables prepared:
column_overview: 25 rows
categorical_unique_values: 270 rows
categorical_summary: 8 rows
numeric_summary: 3 rows
date_summary: 3 rows
binary_columns: 4 rows
high_cardinality_columns: 50 rows
excluded_columns: 11 rows


## 6. Write Excel Workbook

In [6]:
sheets = {
    "README": readme,
    "column_overview": column_overview,
    "categorical_unique_values": categorical_unique_values,
    "categorical_summary": categorical_summary,
    "numeric_summary": numeric_summary,
    "date_summary": date_summary,
    "binary_columns": binary_columns,
    "high_cardinality_columns": high_cardinality_columns,
    "excluded_columns": excluded_columns,
}

def write_with_pandas_excel_writer(path, sheets):
    try:
        import openpyxl  # noqa: F401
        excel_engine = "openpyxl"
    except Exception:
        try:
            import xlsxwriter  # noqa: F401
            excel_engine = "xlsxwriter"
        except Exception:
            return False

    with pd.ExcelWriter(path, engine=excel_engine) as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    return True


def excel_col_name(n):
    name = ""
    while n:
        n, rem = divmod(n - 1, 26)
        name = chr(65 + rem) + name
    return name


def cell_xml(row_idx, col_idx, value):
    import html
    import pandas as pd
    import numpy as np

    cell_ref = f"{excel_col_name(col_idx)}{row_idx}"
    if value is None or value is pd.NA:
        return f'<c r="{cell_ref}"/>'
    if isinstance(value, float) and np.isnan(value):
        return f'<c r="{cell_ref}"/>'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        value = str(pd.to_datetime(value))
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return f'<c r="{cell_ref}"><v>{value}</v></c>'
    text = html.escape(str(value), quote=False)
    return f'<c r="{cell_ref}" t="inlineStr"><is><t>{text}</t></is></c>'


def worksheet_xml(df):
    max_excel_rows = 1_048_576
    if len(df) + 1 > max_excel_rows:
        df = df.head(max_excel_rows - 1).copy()
        df.loc[len(df)] = ["TRUNCATED_TO_EXCEL_ROW_LIMIT"] + [""] * (len(df.columns) - 1)

    rows = []
    header_cells = [cell_xml(1, i + 1, col) for i, col in enumerate(df.columns)]
    rows.append(f'<row r="1">{"".join(header_cells)}</row>')
    for r, row in enumerate(df.itertuples(index=False, name=None), start=2):
        cells = [cell_xml(r, c + 1, value) for c, value in enumerate(row)]
        rows.append(f'<row r="{r}">{"".join(cells)}</row>')
    return '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>' + \
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">' + \
        '<sheetData>' + ''.join(rows) + '</sheetData></worksheet>'


def write_minimal_xlsx(path, sheets):
    import zipfile
    import html

    sheet_items = list(sheets.items())
    content_types = [
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">',
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>',
        '<Default Extension="xml" ContentType="application/xml"/>',
        '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>',
    ]
    for i in range(1, len(sheet_items) + 1):
        content_types.append(f'<Override PartName="/xl/worksheets/sheet{i}.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>')
    content_types.append('</Types>')

    root_rels = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>' \
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">' \
        '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>' \
        '</Relationships>'

    workbook_sheets = []
    workbook_rels = ['<?xml version="1.0" encoding="UTF-8" standalone="yes"?>', '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">']
    for i, (sheet_name, _) in enumerate(sheet_items, start=1):
        safe_name = html.escape(sheet_name[:31], quote=True)
        workbook_sheets.append(f'<sheet name="{safe_name}" sheetId="{i}" r:id="rId{i}"/>')
        workbook_rels.append(f'<Relationship Id="rId{i}" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet{i}.xml"/>')
    workbook_rels.append('</Relationships>')

    workbook_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>' \
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">' \
        '<sheets>' + ''.join(workbook_sheets) + '</sheets></workbook>'

    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('[Content_Types].xml', ''.join(content_types))
        zf.writestr('_rels/.rels', root_rels)
        zf.writestr('xl/workbook.xml', workbook_xml)
        zf.writestr('xl/_rels/workbook.xml.rels', ''.join(workbook_rels))
        for i, (_, df) in enumerate(sheet_items, start=1):
            zf.writestr(f'xl/worksheets/sheet{i}.xml', worksheet_xml(df))


if write_with_pandas_excel_writer(OUTPUT_XLSX, sheets):
    print(f"Excel written with pandas ExcelWriter: {OUTPUT_XLSX}")
else:
    write_minimal_xlsx(OUTPUT_XLSX, sheets)
    print(f"Excel written with built-in fallback writer: {OUTPUT_XLSX}")

Excel written with built-in fallback writer: E:\000硕士毕设\公司选取\UKcompanies_8_sectors_cleaned_label_profile.xlsx
